In [ ]:
%pip install --upgrade pip
%pip install tensorflow notebook ipykernel matplotlib numpy pandas scikit-learn seaborn kagglehub

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# -------------------- CONFIGURATION --------------------
EPOCHS = 15
SEQ_LEN = 40
MAX_CHARS = 200000
BATCH_SIZE = 128
keras.utils.set_random_seed(42)

In [ ]:
# -------------------- GPU / CPU SETUP --------------------
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU available:", gpus[0].name)
else:
    print("GPU not found. Training will run using CPU.")
    print("CPU training may take longer.")

In [ ]:
# -------------------- LOAD DATASET --------------------
path = keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
)
with open(path, encoding="utf-8") as file:
    text = file.read().lower()[:MAX_CHARS]
chars = sorted(set(text))
char_to_id = {c: i for i, c in enumerate(chars)}
id_to_char = np.array(chars)
print("\nTotal characters:", len(text))
print("Vocabulary size:", len(chars))
print("\nSample text:\n", text[:300])

In [ ]:
# -------------------- ENCODE TEXT --------------------
encoded = np.array([char_to_id[c] for c in text], dtype=np.int32)
# Create input sequences and target characters
starts = range(0, len(encoded) - SEQ_LEN, 3)
X = np.array([
    encoded[i:i + SEQ_LEN] for i in starts
], dtype=np.int32)
y = np.array([
    encoded[i + SEQ_LEN] for i in starts
], dtype=np.int32)
print("\nInput shape:", X.shape)
print("Output shape:", y.shape)

In [ ]:
# -------------------- BUILD LSTM MODEL --------------------
model = keras.Sequential([
    layers.Input(shape=(SEQ_LEN,)),
    layers.Embedding(input_dim=len(chars), output_dim=64),
    layers.LSTM(256),
    layers.Dense(len(chars), activation="softmax")
])
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
# -------------------- TRAIN MODEL --------------------
print("\nStarting LSTM training...\n")
history = model.fit(
    X,
    y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    verbose=2
)

In [ ]:
# -------------------- PLOT ACCURACY AND LOSS --------------------
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history["accuracy"], label="Training Accuracy")
ax[0].plot(history.history["val_accuracy"], label="Validation Accuracy")
ax[0].set_title("Model Accuracy")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Accuracy")
ax[0].legend()
ax[1].plot(history.history["loss"], label="Training Loss")
ax[1].plot(history.history["val_loss"], label="Validation Loss")
ax[1].set_title("Model Loss")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Loss")
ax[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# -------------------- EVALUATION --------------------
print("\nFinal Validation Accuracy:",
      round(history.history["val_accuracy"][-1], 4))
print("Validation Perplexity:",
      round(float(np.exp(history.history["val_loss"][-1])), 2))

In [ ]:
# -------------------- TEXT GENERATION --------------------
def generate(seed, n_chars=400, temperature=0.8):
    if temperature <= 0:
        raise ValueError("Temperature must be greater than zero.")
    sentence = seed.lower()
    for _ in range(n_chars):
        context = sentence[-SEQ_LEN:]
        # Pad shorter contexts to the required sequence length
        context = context.rjust(SEQ_LEN)
        x = np.array(
            [[char_to_id[c] for c in context]],
            dtype=np.int32
        )
        predictions = model.predict(x, verbose=0)[0]
        # Temperature-based sampling
        logits = np.log(predictions + 1e-8) / temperature
        logits -= np.max(logits)
        probabilities = np.exp(logits)
        probabilities /= np.sum(probabilities)
        next_char = np.random.choice(
            len(chars),
            p=probabilities
        )
        sentence += id_to_char[next_char]
    return sentence

In [ ]:
# -------------------- GENERATE TEXT --------------------
seed = text[1000:1000 + SEQ_LEN]
print("\n========== GENERATED SHAKESPEARE TEXT ==========")
for temperature in [0.2, 0.5, 1.0]:
    print("\nTemperature:", temperature)
    print(generate(seed, n_chars=400, temperature=temperature))
print("\nLSTM Text Generation Completed Successfully!")